# 01 · Preprocess MC-MED (CPU session)

Runs the data stages: **ingest → featurize → label → split → index**. Each stage is cached by a hash of its
config, so rerunning only recomputes what changed.

**Before running:** attach your *private* MC-MED Kaggle dataset, and set the repo URL in the first cell.
After it finishes: **Save Version → Save & Run All**, then attach this notebook's output to notebook 02.

> PhysioNet DUA: keep the dataset and this notebook private; never print patient-level rows in outputs.

In [ ]:
# --- Setup: install this repo (code only; data comes from the attached Kaggle dataset) ---
# Private repo? Add a GitHub token as a Kaggle Secret named GITHUB_TOKEN (Add-ons > Secrets).
import os, subprocess, sys
REPO = "github.com/<YOUR_GITHUB_USERNAME>/ed-deterioration-forecast.git"   # <- edit once
BRANCH = "main"
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = f"https://{tok}@{REPO}"
except Exception:
    url = f"https://{REPO}"
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/kaggle/working/repo[train]"], check=True)
os.chdir("/kaggle/working/repo")

In [ ]:
# --- Paths: keep RAW identical in every notebook (it is part of the cache key) ---
RAW  = "/kaggle/input/mc-med"                 # <- your PRIVATE Kaggle dataset with MC-MED CSVs
WORK = "/kaggle/working/work"
RUNS = "/kaggle/working/runs"
COMMON = [f"paths.raw_dir={RAW}", f"paths.work_dir={WORK}", f"paths.runs_dir={RUNS}"]
ARGS = " ".join(COMMON)


In [ ]:
# Optional quick check on a subset first:  add "data.max_visits=2000"
!edews run --stages index {ARGS}

In [ ]:
import json, glob
for f in sorted(glob.glob(f"{WORK}/*/*/meta.json")):
    m = json.load(open(f)); print(m["stage"], m["key"], {k: v for k, v in m.items() if k not in ("stage", "key")})